# LeetCode 1710 — Maximum Units on a Truck bằng Python

**Link đề:** https://leetcode.com/problems/maximum-units-on-a-truck/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy, Sorting

```python
class Solution:
    def maximumUnits(self, boxTypes: List[List[int]], truckSize: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu bài toán chọn hộp và nhận ra đây là **tham lam (greedy)**: luôn lấy hộp chứa nhiều unit nhất trước.
2. Nắm công cụ Python: unpacking `[a, b]`, `sorted(..., key=lambda ..., reverse=True)`, `min()`, `heapq`, mảng đếm (bucket).
3. Đi từ cách "trải từng hộp ra rồi sắp xếp" → sắp xếp **theo loại hộp** → **counting sort** theo số unit (`≤ 1000`).
4. Trace từng loại hộp, chứng minh tham lam đúng bằng **lập luận hoán đổi (exchange argument)**.
5. Phân biệt bài này (chia được theo từng hộp) với **knapsack 0/1** (không tham lam được).

## 1. Đề bài và ví dụ

Có `boxTypes[i] = [numberOfBoxes_i, numberOfUnitsPerBox_i]`: loại `i` có `numberOfBoxes_i` hộp, mỗi hộp chứa `numberOfUnitsPerBox_i` unit. Xe tải chở **tối đa `truckSize` hộp**. Được chọn hộp tuỳ ý (miễn tổng số hộp ≤ `truckSize`). Trả về **tổng unit lớn nhất**.

| `boxTypes` | `truckSize` | Kết quả | Giải thích |
|---|:---:|:---:|---|
| `[[1,3],[2,2],[3,1]]` | 4 | `8` | 1 hộp×3 + 2 hộp×2 + 1 hộp×1 = 8 |
| `[[5,10],[2,5],[4,7],[3,9]]` | 10 | `91` | 5×10 + 3×9 + 2×7 = 91 |

**Constraints:**
- `1 <= len(boxTypes) <= 1000`
- `1 <= numberOfBoxes_i, numberOfUnitsPerBox_i <= 1000`
- `1 <= truckSize <= 10^6`

### 1.1. Tính tay ví dụ 2

Sắp loại hộp theo unit/hộp giảm dần: `[5,10]`, `[3,9]`, `[4,7]`, `[2,5]`.
- Lấy 5 hộp loại 10 → còn chỗ 5, được 50.
- Lấy 3 hộp loại 9 → còn chỗ 2, được 27 (tổng 77).
- Loại 7 có 4 hộp nhưng chỉ còn 2 chỗ → lấy 2, được 14 (tổng 91). Xe đầy, dừng.

In [1]:
from typing import List

print("5*10 + 3*9 + 2*7 =", 5 * 10 + 3 * 9 + 2 * 7)
print("Nếu lấy nhầm loại 5 trước: 2*5 + 5*10 + 3*9 =", 2 * 5 + 5 * 10 + 3 * 9, "(ít hơn)")

5*10 + 3*9 + 2*7 = 91
Nếu lấy nhầm loại 5 trước: 2*5 + 5*10 + 3*9 = 87 (ít hơn)


## 2. Nền tảng Python cần biết

### 2.1. Unpacking một cặp `[a, b]`

`boxes, units = [5, 10]` gán `boxes = 5`, `units = 10`. Trong vòng `for`: `for boxes, units in boxTypes:`.

In [2]:
boxTypes = [[1, 3], [2, 2], [3, 1]]
for boxes, units in boxTypes:
    print("loại có", boxes, "hộp, mỗi hộp", units, "unit -> tổng loại", boxes * units)

loại có 1 hộp, mỗi hộp 3 unit -> tổng loại 3
loại có 2 hộp, mỗi hộp 2 unit -> tổng loại 4
loại có 3 hộp, mỗi hộp 1 unit -> tổng loại 3


### 2.2. `sorted` với `key` và `reverse=True`

`sorted(lst, key=f, reverse=True)` trả về **list mới** sắp giảm dần theo `f(x)`. `lst.sort(...)` sắp **tại chỗ** (thay đổi list gốc, trả `None`). Ở đây `key=lambda bt: bt[1]` nghĩa là "sắp theo số unit mỗi hộp".

In [3]:
bt = [[5, 10], [2, 5], [4, 7], [3, 9]]
print("sorted giảm theo unit:", sorted(bt, key=lambda x: x[1], reverse=True))
print("bt gốc không đổi:", bt)
bt2 = bt[:]
r = bt2.sort(key=lambda x: -x[1])        # dấu trừ cũng cho thứ tự giảm
print("sort tại chỗ:", bt2, "| giá trị trả về:", r)

sorted giảm theo unit: [[5, 10], [3, 9], [4, 7], [2, 5]]
bt gốc không đổi: [[5, 10], [2, 5], [4, 7], [3, 9]]
sort tại chỗ: [[5, 10], [3, 9], [4, 7], [2, 5]] | giá trị trả về: None


### 2.3. `min(a, b)` — lấy bao nhiêu hộp?

Với một loại có `boxes` hộp và còn `space` chỗ: số hộp lấy được là `min(boxes, space)`.

In [4]:
for boxes, space in [(5, 10), (4, 2), (3, 3)]:
    print(f"boxes = {boxes}, space = {space} -> lấy {min(boxes, space)}")

boxes = 5, space = 10 -> lấy 5
boxes = 4, space = 2 -> lấy 2
boxes = 3, space = 3 -> lấy 3


### 2.4. `heapq` — hàng đợi ưu tiên (min-heap)

`heapq` giữ list sao cho phần tử **nhỏ nhất** luôn ở `heap[0]`. `heappush` / `heappop` tốn `O(log n)`, `heapify` tốn `O(n)`. Muốn lấy **lớn nhất** trước → lưu **số âm**.

In [5]:
import heapq

h = [(-10, 5), (-5, 2), (-7, 4), (-9, 3)]     # (-units, boxes)
heapq.heapify(h)
while h:
    neg_units, boxes = heapq.heappop(h)
    print("lấy ra loại unit =", -neg_units, "với", boxes, "hộp")

lấy ra loại unit = 10 với 5 hộp
lấy ra loại unit = 9 với 3 hộp
lấy ra loại unit = 7 với 4 hộp
lấy ra loại unit = 5 với 2 hộp


### 2.5. Mảng đếm (bucket) theo giá trị nhỏ

Vì `units ≤ 1000`, có thể tạo `count[u]` = tổng số hộp có `u` unit. Duyệt `u` từ 1000 xuống 1 là đã có thứ tự giảm dần — không cần sắp xếp!

In [6]:
bt = [[5, 10], [2, 5], [4, 7], [3, 9], [1, 10]]
count = [0] * 11                 # chỉ số 0..10 cho ví dụ nhỏ
for boxes, units in bt:
    count[units] += boxes        # hai loại cùng 10 unit được gộp
print("count =", count)
print("duyệt giảm dần:", [(u, count[u]) for u in range(10, 0, -1) if count[u]])

count = [0, 0, 0, 0, 0, 2, 0, 4, 0, 3, 6]
duyệt giảm dần: [(10, 6), (9, 3), (7, 4), (5, 2)]


## 3. Cách trực giác — trải từng hộp ra rồi chọn `truckSize` hộp tốt nhất

Mỗi hộp là một "món", giá trị là số unit. Ta liệt kê **mọi hộp riêng lẻ**, sắp giảm dần, lấy `truckSize` hộp đầu.

In [7]:
def max_units_expand(boxTypes: List[List[int]], truckSize: int) -> int:
    all_boxes = []
    for boxes, units in boxTypes:
        all_boxes.extend([units] * boxes)      # [units, units, ...] boxes lần
    all_boxes.sort(reverse=True)
    return sum(all_boxes[:truckSize])

print([3] * 2, "| extend nối list vào cuối")
print(max_units_expand([[1, 3], [2, 2], [3, 1]], 4))              # 8
print(max_units_expand([[5, 10], [2, 5], [4, 7], [3, 9]], 10))    # 91

[3, 3] | extend nối list vào cuối
8
91


### Chi phí và nút thắt

Tổng số hộp có thể là `1000 loại × 1000 hộp = 10^6` → sắp xếp `10^6` phần tử, bộ nhớ `10^6` số. Vẫn chạy được nhưng **lãng phí**: các hộp cùng loại giống hệt nhau, không cần tách riêng.

**Cải tiến:** sắp xếp **theo loại** (≤ 1000 loại), rồi lấy **cả khối** `min(boxes, space)` hộp mỗi loại.

In [8]:
bt_big = [[1000, u] for u in range(1, 1001)]
print("số loại:", len(bt_big), "| tổng số hộp nếu trải ra:", sum(b_[0] for b_ in bt_big))

số loại: 1000 | tổng số hộp nếu trải ra: 1000000


## 4. Insight — tham lam theo unit/hộp

Mọi hộp chiếm **đúng 1 chỗ** trên xe. Vậy chỗ trống nào cũng nên được lấp bằng hộp **có nhiều unit nhất còn lại**. Không có lý do gì để chọn một hộp ít unit hơn khi vẫn còn hộp nhiều unit hơn chưa lấy.

Quy tắc: **sắp xếp loại hộp theo `units` giảm dần, lấy càng nhiều càng tốt từ mỗi loại, cho tới khi đầy xe.**

(Gợi ý của đề cũng đúng như vậy.) Khác với knapsack 0/1 kinh điển: ở đó món đồ có **kích thước khác nhau**, nên tham lam theo giá trị bị sai. Ở đây mọi hộp cùng kích thước 1.

## 5. Lời giải chính — sắp xếp + lấy theo khối

**Đoạn 1 — sắp xếp:** `order = sorted(boxTypes, key=lambda x: x[1], reverse=True)`.

**Đoạn 2 — duyệt và lấy:** `space = truckSize`, `total = 0`. Với mỗi `boxes, units`:
- `take = min(boxes, space)`;
- `total += take * units`; `space -= take`;
- nếu `space == 0` thì `break` (xe đầy).

**Đoạn 3:** trả `total`.

In [9]:
def max_units_greedy(boxTypes: List[List[int]], truckSize: int) -> int:
    order = sorted(boxTypes, key=lambda x: x[1], reverse=True)   # đoạn 1
    space = truckSize
    total = 0
    for boxes, units in order:                                     # đoạn 2
        take = min(boxes, space)
        total += take * units
        space -= take
        if space == 0:
            break
    return total                                                   # đoạn 3

print(max_units_greedy([[1, 3], [2, 2], [3, 1]], 4))              # 8
print(max_units_greedy([[5, 10], [2, 5], [4, 7], [3, 9]], 10))    # 91

8
91


## 6. TRACE từng bước

Ví dụ 2 sau khi sắp: `[[5,10],[3,9],[4,7],[2,5]]`, `truckSize = 10`.

| Loại | boxes | units | space trước | take | +units | total | space sau |
|---|---:|---:|---:|---:|---:|---:|---:|
| 1 | 5 | 10 | 10 | 5 | 50 | 50 | 5 |
| 2 | 3 | 9 | 5 | 3 | 27 | 77 | 2 |
| 3 | 4 | 7 | 2 | 2 | 14 | 91 | 0 → dừng |
| 4 | 2 | 5 | — | — | — | — | không xét |

In [10]:
def trace_max_units(boxTypes, truckSize):
    order = sorted(boxTypes, key=lambda x: x[1], reverse=True)
    print("sau khi sắp:", order)
    space, total = truckSize, 0
    for boxes, units in order:
        take = min(boxes, space)
        total += take * units
        print(f"  loại [{boxes},{units}]: space {space:>2} -> lấy {take} hộp (+{take * units:>3}) -> total = {total:>3}, space = {space - take}")
        space -= take
        if space == 0:
            print("  xe đầy -> dừng")
            break
    else:
        print("  hết hộp nhưng xe còn chỗ:", space)
    return total

trace_max_units([[5, 10], [2, 5], [4, 7], [3, 9]], 10)
print()
trace_max_units([[1, 3], [2, 2], [3, 1]], 100)

sau khi sắp: [[5, 10], [3, 9], [4, 7], [2, 5]]
  loại [5,10]: space 10 -> lấy 5 hộp (+ 50) -> total =  50, space = 5
  loại [3,9]: space  5 -> lấy 3 hộp (+ 27) -> total =  77, space = 2
  loại [4,7]: space  2 -> lấy 2 hộp (+ 14) -> total =  91, space = 0
  xe đầy -> dừng

sau khi sắp: [[1, 3], [2, 2], [3, 1]]
  loại [1,3]: space 100 -> lấy 1 hộp (+  3) -> total =   3, space = 99
  loại [2,2]: space 99 -> lấy 2 hộp (+  4) -> total =   7, space = 97
  loại [3,1]: space 97 -> lấy 3 hộp (+  3) -> total =  10, space = 94
  hết hộp nhưng xe còn chỗ: 94


10

## 7. Các lời giải khác

### 7.1. Counting sort theo số unit (`units ≤ 1000`)

Gộp hộp theo số unit vào mảng `count[0..1000]`, duyệt từ 1000 xuống 1. Không cần sắp xếp so sánh.

In [11]:
def max_units_bucket(boxTypes: List[List[int]], truckSize: int) -> int:
    count = [0] * 1001
    for boxes, units in boxTypes:
        count[units] += boxes
    space, total = truckSize, 0
    for units in range(1000, 0, -1):
        if count[units] == 0:
            continue
        take = min(count[units], space)
        total += take * units
        space -= take
        if space == 0:
            break
    return total

print(max_units_bucket([[1, 3], [2, 2], [3, 1]], 4), max_units_bucket([[5, 10], [2, 5], [4, 7], [3, 9]], 10))

8 91


### 7.2. Heap (hàng đợi ưu tiên)

Đưa mọi loại vào max-heap (lưu `-units`), lấy dần loại tốt nhất. Có lợi khi xe **nhỏ** so với số loại: chỉ cần pop vài lần, `heapify` là `O(m)`.

In [12]:
def max_units_heap(boxTypes: List[List[int]], truckSize: int) -> int:
    h = [(-units, boxes) for boxes, units in boxTypes]
    heapq.heapify(h)
    space, total = truckSize, 0
    while h and space > 0:
        neg_units, boxes = heapq.heappop(h)
        take = min(boxes, space)
        total += take * (-neg_units)
        space -= take
    return total

print(max_units_heap([[1, 3], [2, 2], [3, 1]], 4), max_units_heap([[5, 10], [2, 5], [4, 7], [3, 9]], 10))

8 91


### 7.3. Brute force thật sự (chỉ để kiểm chứng, dữ liệu rất nhỏ)

Thử **mọi cách chọn số hộp** `c_i ∈ [0, boxes_i]` cho từng loại, giữ những cách có tổng ≤ `truckSize`. Dùng `itertools.product` để sinh mọi tổ hợp. Số tổ hợp tăng theo cấp số nhân — chỉ dùng để **kiểm chứng tham lam**.

In [13]:
from itertools import product

print("product(range(2), range(3)) =", list(product(range(2), range(3))))

def max_units_bruteforce(boxTypes, truckSize):
    best = 0
    ranges = [range(boxes + 1) for boxes, _ in boxTypes]
    for counts in product(*ranges):          # *ranges: "trải" list thành nhiều đối số
        if sum(counts) <= truckSize:
            value = sum(c * u for c, (_, u) in zip(counts, boxTypes))
            best = max(best, value)
    return best

print(max_units_bruteforce([[1, 3], [2, 2], [3, 1]], 4), max_units_bruteforce([[5, 10], [2, 5], [4, 7], [3, 9]], 10))

product(range(2), range(3)) = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]
8 91


### 7.4. Bảng so sánh

`m = len(boxTypes)`, `U = 1000` (giới hạn units), `T = truckSize`, `N = tổng số hộp`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Brute force tổ hợp | thử mọi `c_i` | `∏(boxes_i + 1)` | `O(m)` | chỉ để kiểm chứng |
| Trải hộp + sort | mỗi hộp là 1 món | `O(N log N)` | `O(N)` | `N` tới `10^6` |
| **Sort theo loại + tham lam** | lấy cả khối | `O(m log m)` | `O(m)` | khuyên dùng |
| Counting sort | `count[units]` | `O(m + U)` | `O(U)` | nhanh khi `U` nhỏ |
| Heap | pop loại tốt nhất | `O(m + k log m)` | `O(m)` | `k` = số loại được lấy |

## 8. Code nộp LeetCode

In [14]:
class Solution:
    def maximumUnits(self, boxTypes: List[List[int]], truckSize: int) -> int:
        boxTypes.sort(key=lambda x: x[1], reverse=True)
        total = 0
        for boxes, units in boxTypes:
            take = min(boxes, truckSize)
            total += take * units
            truckSize -= take
            if truckSize == 0:
                break
        return total

print(Solution().maximumUnits([[1, 3], [2, 2], [3, 1]], 4), Solution().maximumUnits([[5, 10], [2, 5], [4, 7], [3, 9]], 10))

8 91


Lưu ý: bản này dùng `boxTypes.sort` **tại chỗ** (sửa input) và dùng luôn `truckSize` làm biến "chỗ còn lại" — chấp nhận được trên LeetCode. Trong code thật, nên dùng `sorted` để không làm thay đổi dữ liệu của người gọi.

Bản counting sort để nộp:

In [15]:
class SolutionBucket:
    def maximumUnits(self, boxTypes: List[List[int]], truckSize: int) -> int:
        count = [0] * 1001
        for boxes, units in boxTypes:
            count[units] += boxes
        total = 0
        for units in range(1000, 0, -1):
            if count[units]:
                take = min(count[units], truckSize)
                total += take * units
                truckSize -= take
                if truckSize == 0:
                    break
        return total

print(SolutionBucket().maximumUnits([[5, 10], [2, 5], [4, 7], [3, 9]], 10))

91


## 9. Vì sao tham lam đúng? — lập luận hoán đổi

Gọi **G** là tập hộp tham lam chọn, **O** là một lời giải tối ưu bất kỳ. Cả hai đều có `min(T, N)` hộp (lấy thêm hộp không bao giờ làm giảm unit, vì unit ≥ 1).

Giả sử `O ≠ G`. Khi đó tồn tại hộp `x ∈ G \ O` và hộp `y ∈ O \ G`. Vì tham lam lấy hộp theo unit giảm dần mà **không** lấy `y` nhưng lấy `x`, nên `units(x) ≥ units(y)`. Thay `y` bằng `x` trong `O`: số hộp không đổi, tổng unit **không giảm**. Lặp lại, ta biến `O` thành `G` mà không bao giờ giảm giá trị ⇒ `value(G) ≥ value(O)` ⇒ tham lam tối ưu.

**Invariant của vòng lặp:** sau khi xét `j` loại đầu (đã sắp), `total` = tổng unit của `truckSize_ban_đầu − space` hộp tốt nhất trong toàn bộ dữ liệu.

## 10. Độ phức tạp

`m = len(boxTypes) ≤ 1000`.
- **Sort + tham lam:** sắp xếp `O(m log m)`, duyệt `O(m)` ⇒ **Time `O(m log m)`**. Space `O(m)` cho `sorted` (hoặc `O(1)` phụ nếu sort tại chỗ, Timsort dùng tối đa `O(m)` bộ nhớ tạm).
- **Counting sort:** `O(m + 1000)` Time, `O(1000)` Space.
- Không phụ thuộc `truckSize` (có thể tới `10^6`) — ta lấy theo **khối**, không lặp từng hộp.

In [16]:
import random, time

random.seed(1710)
bt_rand = [[random.randint(1, 1000), random.randint(1, 1000)] for _ in range(1000)]
T = 10**6
for name, f in [("expand", max_units_expand), ("greedy", max_units_greedy),
                ("bucket", max_units_bucket), ("heap", max_units_heap)]:
    t = time.perf_counter()
    r = f([x[:] for x in bt_rand], T)
    print(f"{name:>7}: {r}, {time.perf_counter() - t:.4f}s")

 expand: 249231061, 0.0201s
 greedy: 249231061, 0.0004s
 bucket: 249231061, 0.0002s
   heap: 249231061, 0.0005s


## 11. Bộ test

1. Hai ví dụ đề.
2. Biên: 1 loại, xe chở ít hơn / nhiều hơn số hộp; `truckSize` rất lớn (lấy hết).
3. Dễ sai: nhiều loại **cùng** units; loại tốt nhất có nhiều hộp hơn chỗ trống; đầu vào chưa sắp xếp.
4. Đối chiếu brute force tổ hợp trên dữ liệu nhỏ, và các cách nhanh với nhau trên dữ liệu lớn.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

def solve(bt, T):
    return Solution().maximumUnits([x[:] for x in bt], T)

check("đề 1", solve([[1, 3], [2, 2], [3, 1]], 4), 8)
check("đề 2", solve([[5, 10], [2, 5], [4, 7], [3, 9]], 10), 91)
check("1 loại, xe nhỏ", solve([[10, 7]], 3), 21)
check("1 loại, xe lớn", solve([[2, 7]], 100), 14)
check("xe rất lớn lấy hết", solve([[1, 3], [2, 2], [3, 1]], 10**6), 10)
check("cùng units", solve([[2, 5], [3, 5], [1, 1]], 4), 20)
check("loại tốt nhất dư hộp", solve([[100, 9], [1, 1]], 5), 45)

PASS đề 1: 8
PASS đề 2: 91
PASS 1 loại, xe nhỏ: 21
PASS 1 loại, xe lớn: 14
PASS xe rất lớn lấy hết: 10
PASS cùng units: 20
PASS loại tốt nhất dư hộp: 45


In [18]:
random.seed(7)
for _ in range(400):
    m = random.randint(1, 4)
    bt = [[random.randint(1, 3), random.randint(1, 10)] for _ in range(m)]
    T = random.randint(1, 12)
    expected = max_units_bruteforce(bt, T)
    for f in [max_units_expand, max_units_greedy, max_units_bucket, max_units_heap,
              Solution().maximumUnits, SolutionBucket().maximumUnits]:
        assert f([x[:] for x in bt], T) == expected, (f, bt, T)
print("Tham lam khớp brute force trên 400 bộ nhỏ: PASS")

for _ in range(50):
    bt = [[random.randint(1, 1000), random.randint(1, 1000)] for _ in range(random.randint(1, 300))]
    T = random.randint(1, 10**6)
    vals = {f([x[:] for x in bt], T) for f in [max_units_greedy, max_units_bucket, max_units_heap, Solution().maximumUnits]}
    assert len(vals) == 1
print("Các cách nhanh khớp nhau trên 50 bộ lớn: PASS")

Tham lam khớp brute force trên 400 bộ nhỏ: PASS
Các cách nhanh khớp nhau trên 50 bộ lớn: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp theo **số hộp** thay vì **unit/hộp**

In [19]:
def wrong_sort_by_boxes(bt, T):
    order = sorted(bt, key=lambda x: x[0], reverse=True)   # SAI: x[0] là số hộp
    total = 0
    for boxes, units in order:
        take = min(boxes, T); total += take * units; T -= take
    return total

bt = [[5, 10], [2, 5], [4, 7], [3, 9]]
print("sai:", wrong_sort_by_boxes(bt, 10), "| đúng:", max_units_greedy(bt, 10))

sai: 87 | đúng: 91


**Sửa:** `key=lambda x: x[1]` — sắp theo phần tử thứ hai (units mỗi hộp).

### Lỗi 2 — Sắp theo **tổng unit của loại** (`boxes * units`)

Loại có tổng lớn chưa chắc tốt **từng hộp**: `[1000, 1]` (tổng 1000) sẽ xếp trước `[1, 999]` (tổng 999), nhưng nếu xe chỉ chở 1 hộp thì hộp 999 unit tốt hơn hẳn.

In [20]:
def wrong_sort_by_total(bt, T):
    order = sorted(bt, key=lambda x: x[0] * x[1], reverse=True)   # SAI
    total = 0
    for boxes, units in order:
        take = min(boxes, T); total += take * units; T -= take
    return total

bt = [[1000, 1], [1, 999]]
print("sai:", wrong_sort_by_total(bt, 1), "| đúng:", max_units_greedy(bt, 1))

sai: 1 | đúng: 999


**Sửa:** mỗi hộp chiếm 1 chỗ ⇒ tiêu chí đúng là **unit trên mỗi hộp**.

### Lỗi 3 — Lấy cả loại mà không giới hạn bởi chỗ trống

In [21]:
def wrong_no_min(bt, T):
    order = sorted(bt, key=lambda x: x[1], reverse=True)
    total = 0
    for boxes, units in order:
        if T <= 0:
            break
        total += boxes * units          # SAI: có thể lấy quá chỗ
        T -= boxes
    return total

print("sai:", wrong_no_min([[5, 10], [2, 5], [4, 7], [3, 9]], 10), "| đúng:", max_units_greedy([[5, 10], [2, 5], [4, 7], [3, 9]], 10))

sai: 105 | đúng: 91


**Sửa:** `take = min(boxes, space)`.

### Lỗi 4 — Lặp từng hộp một (chậm khi `truckSize = 10^6`)

`for _ in range(take): total += units` đúng nhưng tốn `O(truckSize)`. **Sửa:** cộng một lần `take * units`.

## 13. Mẹo nhận dạng pattern

- "Chọn tối đa `K` món, mỗi món **cùng kích thước**, tối đa tổng giá trị" → **tham lam theo giá trị**, sắp giảm dần.
- Món có **kích thước khác nhau** nhưng **chia nhỏ được** (fractional knapsack) → tham lam theo **giá trị/kích thước**.
- Món kích thước khác nhau và **không chia được** (0/1 knapsack) → tham lam **sai**, cần **quy hoạch động**.
- Giá trị nhỏ, bị chặn (≤ 1000) → có thể **counting sort** thay cho sort so sánh.
- Bài liên quan: 1005 Maximize Sum Of Array After K Negations, 2279 Maximum Bags With Full Capacity of Rocks, 455 Assign Cookies, 881 Boats to Save People.

## 14. Biến thể

### Biến thể 1 — Fractional knapsack: hộp có **khối lượng khác nhau**, được cắt nhỏ

`items = [(weight, value)]`, sức chứa `W`. Tham lam theo `value / weight`.

In [22]:
def fractional_knapsack(items, W):
    total = 0.0
    for w, v in sorted(items, key=lambda x: x[1] / x[0], reverse=True):
        take = min(w, W)
        total += v * take / w
        W -= take
        if W == 0:
            break
    return total

print(fractional_knapsack([(10, 60), (20, 100), (30, 120)], 50))   # 240.0

240.0


### Biến thể 2 — Phản ví dụ: 0/1 knapsack, tham lam **sai**

Mỗi món lấy nguyên hoặc không lấy. Tham lam theo `value/weight` ra 160, nhưng tối ưu là 220.

In [23]:
def knapsack_01_greedy(items, W):
    total = 0
    for w, v in sorted(items, key=lambda x: x[1] / x[0], reverse=True):
        if w <= W:
            total += v; W -= w
    return total

def knapsack_01_dp(items, W):
    dp = [0] * (W + 1)                  # dp[c] = giá trị lớn nhất với sức chứa c
    for w, v in items:
        for c in range(W, w - 1, -1):   # duyệt ngược để mỗi món dùng 1 lần
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[W]

items = [(10, 60), (20, 100), (30, 120)]
print("tham lam:", knapsack_01_greedy(items, 50), "| quy hoạch động:", knapsack_01_dp(items, 50))

tham lam: 160 | quy hoạch động: 220


### Biến thể 3 — Trả về **danh sách hộp** đã lấy theo từng loại (giữ chỉ số gốc)

Dùng `sorted(range(m), key=...)` để sắp **chỉ số** thay vì sắp dữ liệu.

In [24]:
def chosen_boxes(boxTypes, truckSize):
    order = sorted(range(len(boxTypes)), key=lambda i: boxTypes[i][1], reverse=True)
    taken = [0] * len(boxTypes)
    for i in order:
        take = min(boxTypes[i][0], truckSize)
        taken[i] = take
        truckSize -= take
        if truckSize == 0:
            break
    return taken

print(chosen_boxes([[5, 10], [2, 5], [4, 7], [3, 9]], 10))   # [5, 0, 2, 3]

[5, 0, 2, 3]


## 15. Bài tập (tăng dần)

> Hàm bài tập chỉ có `pass`, **không được gọi** khi chạy notebook.

### Bài 1 — Tính tay
`boxTypes = [[2, 1], [4, 4], [3, 3]]`, `truckSize = 6`. Đáp án bao nhiêu?

In [25]:
def exercise_1() -> int:
    # TODO
    pass

# assert exercise_1() == Solution().maximumUnits([[2, 1], [4, 4], [3, 3]], 6)

### Bài 2 — Điền chỗ trống lời giải tham lam

In [26]:
def greedy_fill(boxTypes, truckSize):
    # TODO 1: sắp xếp giảm dần theo units
    # TODO 2: duyệt, take = min(...), cộng take * units, giảm truckSize
    # TODO 3: dừng khi đầy, trả total
    pass

# assert greedy_fill([[5, 10], [2, 5], [4, 7], [3, 9]], 10) == 91

### Bài 3 — Số hộp thực sự được chở
Trả về tổng số hộp lên xe (≤ `truckSize`).

In [27]:
def boxes_loaded(boxTypes, truckSize):
    # TODO
    pass

# assert boxes_loaded([[1, 3], [2, 2], [3, 1]], 4) == 4
# assert boxes_loaded([[1, 3], [2, 2], [3, 1]], 100) == 6

### Bài 4 — Tự viết bản counting sort

In [28]:
def bucket_rewrite(boxTypes, truckSize):
    # TODO: count[units] += boxes, duyệt units từ lớn xuống nhỏ
    pass

# assert bucket_rewrite([[5, 10], [2, 5], [4, 7], [3, 9]], 10) == 91

### Bài 5 — Biến thể: hai xe tải
Có **hai** xe, sức chứa `t1` và `t2` hộp. Tổng unit lớn nhất? (Gợi ý: có khác một xe sức chứa `t1 + t2` không?)

In [29]:
def two_trucks(boxTypes, t1, t2):
    # TODO
    pass

# assert two_trucks([[5, 10], [2, 5], [4, 7], [3, 9]], 4, 6) == 91

### Bài 6 — Mini challenge: xe nhỏ nhất đạt tối thiểu `target` unit
Tìm `truckSize` **nhỏ nhất** để chở được ít nhất `target` unit; trả `-1` nếu không thể.

In [30]:
def min_truck_for_target(boxTypes, target):
    # TODO: lấy tham lam từng khối; khi khối hiện tại đủ, tính số hộp cần bằng phép chia làm tròn lên
    pass

# assert min_truck_for_target([[5, 10], [2, 5], [4, 7], [3, 9]], 91) == 10
# assert min_truck_for_target([[1, 3]], 4) == -1

## 16. Gợi ý 3 mức

**Bài 1:** (1) Sắp: `[4,4]`, `[3,3]`, `[2,1]`. (2) Lấy 4 hộp loại 4, còn 2 chỗ. (3) `16 + 2*3 = 22`.

**Bài 2:** (1) Như mục 5. (2) `sorted(..., key=lambda x: x[1], reverse=True)`. (3) `if truckSize == 0: break`.

**Bài 3:** (1) Hộp lên xe = `min(tổng số hộp, truckSize)`. (2) Hoặc cộng dồn `take`. (3) `min(sum(b for b, _ in boxTypes), truckSize)`.

**Bài 4:** (1) `count = [0] * 1001`. (2) `range(1000, 0, -1)`. (3) Như `max_units_bucket`.

**Bài 5:** (1) Hộp không bị ràng buộc xe nào. (2) Hai xe ≡ một xe `t1 + t2` hộp. (3) `max_units_greedy(boxTypes, t1 + t2)`.

**Bài 6:** (1) Duyệt theo units giảm dần, giữ `need = target − total`. (2) Nếu `boxes * units >= need` thì cần `ceil(need / units)` hộp nữa. (3) `-(-need // units)` là chia làm tròn lên.

## 17. Đáp án tham khảo

In [31]:
def ans_exercise_1():
    return 4 * 4 + 2 * 3

def ans_greedy_fill(boxTypes, truckSize):
    total = 0
    for boxes, units in sorted(boxTypes, key=lambda x: x[1], reverse=True):
        take = min(boxes, truckSize)
        total += take * units
        truckSize -= take
        if truckSize == 0:
            break
    return total

def ans_boxes_loaded(boxTypes, truckSize):
    return min(sum(b_ for b_, _ in boxTypes), truckSize)

def ans_bucket_rewrite(boxTypes, truckSize):
    count = [0] * 1001
    for boxes, units in boxTypes:
        count[units] += boxes
    total = 0
    for units in range(1000, 0, -1):
        take = min(count[units], truckSize)
        total += take * units
        truckSize -= take
        if truckSize == 0:
            break
    return total

def ans_two_trucks(boxTypes, t1, t2):
    return ans_greedy_fill(boxTypes, t1 + t2)

def ans_min_truck_for_target(boxTypes, target):
    used, total = 0, 0
    for boxes, units in sorted(boxTypes, key=lambda x: x[1], reverse=True):
        need = target - total
        if need <= 0:
            return used
        if boxes * units >= need:
            return used + (-(-need // units))
        used += boxes
        total += boxes * units
    return used if total >= target else -1

assert ans_exercise_1() == Solution().maximumUnits([[2, 1], [4, 4], [3, 3]], 6)
assert ans_greedy_fill([[5, 10], [2, 5], [4, 7], [3, 9]], 10) == 91
assert ans_boxes_loaded([[1, 3], [2, 2], [3, 1]], 4) == 4 and ans_boxes_loaded([[1, 3], [2, 2], [3, 1]], 100) == 6
assert ans_bucket_rewrite([[5, 10], [2, 5], [4, 7], [3, 9]], 10) == 91
assert ans_two_trucks([[5, 10], [2, 5], [4, 7], [3, 9]], 4, 6) == 91
assert ans_min_truck_for_target([[5, 10], [2, 5], [4, 7], [3, 9]], 91) == 10
assert ans_min_truck_for_target([[1, 3]], 4) == -1
# kiểm tra Bài 6 bằng cách thử mọi truckSize
for _ in range(300):
    bt = [[random.randint(1, 5), random.randint(1, 9)] for _ in range(random.randint(1, 4))]
    target = random.randint(1, 120)
    total_boxes = sum(x[0] for x in bt)
    brute = next((T for T in range(1, total_boxes + 1) if max_units_greedy(bt, T) >= target), -1)
    assert ans_min_truck_for_target(bt, target) == brute, (bt, target)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maximumUnits(self, boxTypes, truckSize):
        total = 0
        for boxes, units in sorted(boxTypes, key=lambda x: x[1], reverse=True):  # unit/hộp giảm dần
            take = min(boxes, truckSize)       # lấy cả khối, không vượt chỗ
            total += take * units
            truckSize -= take
            if truckSize == 0:
                break
        return total
```

**Khung tư duy:** mọi món cùng kích thước → sắp theo giá trị giảm dần → lấy tham lam theo khối → dừng khi đầy.

**Checklist trước khi submit**
- [ ] Sắp theo `x[1]` (unit mỗi hộp), **giảm dần**?
- [ ] `take = min(boxes, space)` — không lấy quá chỗ?
- [ ] Cộng `take * units` một lần (không lặp từng hộp)?
- [ ] Trường hợp xe lớn hơn tổng số hộp: vòng lặp kết thúc tự nhiên, trả đúng tổng?
- [ ] Tên method `maximumUnits`, trả về `int`?